In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from darts.metrics import mape, rmse, mae
from darts.utils.statistics import plot_acf

In [7]:
# Generate synthetic time series data
np.random.seed(22)
time = pd.date_range(start="2023-01-01", periods=100, freq="D")
data = 10 + 0.5 * np.arange(100) + 5 * np.sin(2 * np.pi * np.arange(100) / 30) + np.random.normal(scale=2, size=100)
df = pd.DataFrame({"Date": time, "Value": data})
# Create a TimeSeries object
series = TimeSeries.from_dataframe(df, time_col="Date", value_cols="Value")



# Split the data into train and test sets
train, test = series.split_before(0.8)
# Initialize the N-BEATS model
model = NBEATSModel(input_chunk_length=30, output_chunk_length=10, n_epochs=50, random_state=42)
# Fit the model
model.fit(train)
# Perform backtesting
backtest_results = model.backtest(
    series,
    start=0.8,  # Use 80% of the data for the first training
    forecast_horizon=10,
    stride=1,
    retrain=False,  # We've already fitted the model, so we don't need to retrain
    verbose=True,
    metric=[mape, rmse, mae],  # Using multiple metrics
)

print("Finish Training")



# Print the backtest results
print("Backtest Results:")
print(f"MAPE: {backtest_results[0]:.2f}%")
print(f"RMSE: {backtest_results[1]:.2f}")
print(f"MAE: {backtest_results[2]:.2f}")
# Generate historical forecasts for plotting
historical_forecasts = model.historical_forecasts(
    series,
    start=0.8,
    forecast_horizon=10,
    stride=1,
    retrain=False,
    verbose=True,
)
print("Finish Predict")


# Plot the results
plt.figure(figsize=(12, 6))
series.plot(label="Actual")
historical_forecasts.plot(label="Forecast")
plt.title("N-BEATS - Historical Forecasts")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig("NBEATS_Backtest.png")
plt.show()
